# Project Setup and Initialization
Initialize new project. 

In this tutorial we will run a full dosimetry pipeline over the SNNMI Dosimetry Challenge Dataset.

This dataset contains anonymized CT and SPECT DICOM images suitable for testing segmentation and dosimetry workflows. Data is sourced from the University of Michigan Deep Blue repository (DOI: 10.7302/864r-tb45).

In [ ]:
from pytheranostics.data_fetchers import fetch_snmmi_dosimetry_challenge
from pytheranostics.imaging_ds.dicom_ingest import auto_setup_dosimetry_study_inventory
from pytheranostics.imaging_ds import create_studies_with_masks
from pytheranostics.segmentation import totalseg_segment, convert_masks_to_rtstruct
from pytheranostics.dosimetry import build_roi_fit_config
from pytheranostics.dosimetry.voxel_s_dosimetry import VoxelSDosimetry
from pathlib import Path
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

# Ensure a handler exists and is set to INFO
if not root.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(levelname)s:%(name)s:%(message)s")
    handler.setFormatter(formatter)
    root.addHandler(handler)
else:
    for h in root.handlers:
        h.setLevel(logging.INFO)


%reload_ext autoreload
%autoreload 2

## Step 0: Initialize a New Project

Before starting, you can use PyTheranostics' project initialization tool to set up a standardized project structure with configuration templates:

In [ ]:
from pytheranostics import init_project

# Create a new project with all templates and standard directories
project_base = Path("./snmmi_dosimetry_challenge_project")
init_project(project_base)

## Step 1: Download Example Data

We'll use the SNMMI Dosimetry Challenge dataset from University of Michigan Deep Blue. This contains multi-timepoint SPECT/CT data and we will focus on Patient_004.

In [ ]:
fetch_snmmi_dosimetry_challenge(data_home=str(project_base))

## Step 2: Segmentation using TotalSegmentator.

Please take a look at the Total Segmentator Tutorial for more details (add link).

### 2.1: Set up folders for where TotalSegmentator will write segmentation masks and where RT-STRUCT files will be saved.

In [ ]:
# Provide a list of ROOT folder where all CT series under it will be discovered automatically.
# In this case, as an example, we provide the path to a single CT series at the first scan of Patient_004 from the downloaded dataset.
project_data_dirs = [project_base / f"snmmi_dose_challenge/Patient_004/SPECT_Cts/scan{i}/ct" for i in range(1, 5)]

# Specify paths of Where to write segmentations and RT-STRUCT outputs (will be grouped by PatientID)
totalsegmentator_output_dir = project_base / "Segmentations"
rtstruct_output_dir = project_base / "rtstructs"

### 2.2: Run TotalSegmentator

In [ ]:
# Run segmentation on each time point and save results in the specified output directory 
seg_results = []

for project_data_dir in project_data_dirs:

    seg_result = totalseg_segment(
        root_dir=str(project_data_dir),
        base_output_dir=str(totalsegmentator_output_dir),
        device="gpu", # Change to "gpu" if using an NVIDIA GPU, or "cpu" to run on CPU. If GPU, set number of workers to 1 and parallel=False.
        parallel=False,
        max_workers=1, # Number of parallel workers to use if parallel=True
    )
    
    seg_results.append(seg_result)

### 2.3: Select organs at risk and convert binary masks to RTStruct format.

#### Using Configuration Files

By default, converting all 104 structures segmented by TotalSegmentator would create a very large RT-STRUCT file. Configuration files let you:
- **Filter**: Include only the organs you need (e.g., kidneys, liver, spleen)
- **Rename**: Change organ names to match your workflow conventions
- **Combine**: Merge multiple structures into one (e.g., all ribs → "ribs")


For details on configuration files, please refer to the TotalSegmentator tutorial. For this example, we will modify the `total_seg_config.json` with the following content:


```json
{
  "vois": [
    {"voi_name": "kidney_left", "include": true, "new_name": "L Kidney"},
    {"voi_name": "kidney_right", "include": true, "new_name": "R Kidney"},
    {"voi_name": "liver", "include": true, "new_name": "Liver"},
    {"voi_name": "spleen", "include": true, "new_name": "Spleen"}
  ],
}
```

This configuration creates an RT-STRUCT with:
- **L Kidney**
- **R Kidney**
- **Liver**
- **Spleen**

In [ ]:
# Convert segmentation masks to RT-STRUCT format for use in dosimetry calculations
rtstruct_results = []
for seg_result in seg_results:
    rtstruct_result = convert_masks_to_rtstruct(
        segmentation_base_dir=str(totalsegmentator_output_dir),
        ct_series_paths=seg_result["ct_paths"],
        rtstruct_output_dir=str(rtstruct_output_dir ),
        config_path=project_base / "total_seg_config.json",  # Config file specifying which structures to include
    )
    rtstruct_results.append(rtstruct_result)
    

## Step 3: Auto-organize data 

See Data_Ingestion_Examples tutorial for details

### 3.0: Move freshly generated rtstruct files into the imaging data folder

 In this tutorial, SPECT and CT data lies under `project_folder/snmmi_dose_challenge/Patient_004`. The RTStruct files generated in the previous step are stored under `project_folder/rtstructs`. Please move the RTstruct files under the patient folder so that the auto-setup inventory works.

### 3.1: Run Auto-setup / study inventory

In [ ]:
# Organize and extract metadata
study_info, ct_paths, spect_paths, rtstruct_files = auto_setup_dosimetry_study_inventory(
    base_dir=project_base, # / "snmmi_dose_challenge/Patient_004",  # Base directory to search for CT and SPECT series
    patient_id=None,   # auto-detect from DICOM headers
)

# Quick summary
print('Patient ID:', study_info.get('patient_id'))
inj = study_info.get('injection_info', {})
print('Injection date:', inj.get('injection_date'), 'time:', inj.get('injection_time'))
print(f'CT time points:   {len(ct_paths)}')
print(f'SPECT time points: {len(spect_paths)}')
print(f'RTSTRUCT files:    {len(rtstruct_files)}')

# Example: show first CT/SPECT time point folders (if present)
print('First CT tp:', ct_paths[0] if ct_paths else None)
print('First SPECT tp:', spect_paths[0] if spect_paths else None)
print('First RTSTRUCT:', rtstruct_files[0] if rtstruct_files else None)

## Step 3: Set-up VOI Mapping and Fit/Dosimetry Parameters.

Please visit ROI_Mapping_Tutorial for details (add link)

### 3.1: Modify the `dosimetry_fit_defaults.json` to specify default fit parameters.

```JSON
{
  "_description": "Apply bi-exponential fitting with uptake phase to all organs by default. Users can override these defaults for specific organs or ROIs by providing a custom configuration file.",
  "organ_defaults": {
    "fit_order": 2,
    "with_uptake": true,
    "param_init": {"A1": 100, "A2": 0.01},
    "bounds": {"A2": [0.004345, 1.0]},
    "fixed_parameters": null,
    "bounds": null,
    "washout_ratio": null
  }
}
```


### 3.2: Modify VOI Mapping to map names of RTStruct ROIs to standard names in pyTheranostics. Modify the key `ct_mappings` and `spect_mappings` inside `voi_mappings_config.json` to map the RTstruct names stored in segmented regions to pyTheranostics standardized names:

```JSON
"ct_mappings": {
    "LKidney": "Kidney_Left",
      "RKidney": "Kidney_Right",
      "Liver": "Liver",
      "Spleen": "Spleen"
  },

  "spect_mappings": {
    "LKidney": "Kidney_Left",
      "RKidney": "Kidney_Right",
      "Liver": "Liver",
      "Spleen": "Spleen"
  }

  ```

### 3.3: Initialize Dosimetry configuration.

In this step, we build the main configuration dictionary. It is composed of:
- ROIs fitting properties (obtained from the default `dosimetry_fit_defaults.json` or specified by the user)
- Patient ID and Injection information, that can be obtained from the `study_info` dictionary or specified by the user
- Dosimetry Level (Voxel) and method (Voxel-S)
- Since we are running voxel-S convolution method, we can specify if we scale voxel dose by density.
- Reference time point to calculate time-integrated activity at the voxel level.

### 3.3: Load the Data and initialize configuration

In [ ]:
# Load data AND apply mappings in one step
longCT, longSPECT, inj, used_mappings = create_studies_with_masks(
     patient_id=study_info["patient_id"],
     cycle_no=1,
     parallel=True,
     mapping_config=project_base / "voi_mappings_config.json",
     study_info=study_info
 )

# Initilization of roi configuration 
roi_config = build_roi_fit_config(longSPECT=longSPECT, config_path=project_base / "dosimetry_fit_defaults.json")

# Build Dosimetry config
dosimetry_config = {
    "PatientID": study_info["patient_id"],
    "DatabaseDir": str(project_base / "dosimetry_database"),
    
    "VOIs": roi_config,
    
    "InjectionDate": study_info["time_points"][0]["injection_info"]["injection_date"],
    "InjectionTime": study_info["time_points"][0]["injection_info"]["injection_time"],
    "InjectedActivity": longSPECT.meta[0].Injected_Activity_MBq,
    "Radionuclide": longSPECT.meta[0].Radionuclide,
    "PatientWeight_g": study_info["time_points"][0]["injection_info"]["patient_weight_g"],
    
    
    "Level": "Voxel",
    "Method": "Voxel-S-value",
    "ScaleDoseByDensity": False,
    
    "ReferenceTimePoint": 0
}


### 3.4: Initialize Dosimetry Calculator

In [ ]:
Dosimetry = VoxelSDosimetry(config=dosimetry_config, nm_data=longSPECT, ct_data=longCT)

### 3.5 Compute Dose

In [ ]:
Dosimetry.compute_dose()

### 3.6 Review Results

Time-activity data, time-integrated activity (residence times) at the region level at stored under `Dosimetry.results`. Mean dose per region (derived from Voxelized dose map) is summaried in `Dosimetry.df_ad`

In [ ]:
# View Time-Activity data, parameters of fit, etc.
Dosimetry.results

In [ ]:
# View Average Dose
Dosimetry.df_ad